# NB02 – Captioning  (v2, BLIP-2)

Captions every real image with **BLIP-2** (`Salesforce/blip2-opt-2.7b`), cleans and
length-caps each caption, and stores them in `captions/*.parquet`.

**Run in a FRESH Kaggle session** (Run → Factory reset → Run All) to avoid any
package-version pollution from earlier attempts.

- Internet **ON**, GPU **REQUIRED** (T4, ~7 GB VRAM).
- This notebook does **not** upgrade any Kaggle packages.

In [1]:
import importlib.util, sys, subprocess
# Kaggle already ships transformers, torch, Pillow, pyarrow, huggingface_hub.
# Only install genuinely missing packages – NEVER -U.
_need = [p for m,p in [("torch","torch"),("transformers","transformers"),
                        ("PIL","pillow"),("pyarrow","pyarrow"),
                        ("huggingface_hub","huggingface_hub"),
                        ("accelerate","accelerate")]
         if importlib.util.find_spec(m) is None]
if _need:
    subprocess.run([sys.executable,"-m","pip","install","-q",*_need], check=True)
    print("installed:", _need)
else:
    print("all packages already present – no installs, no upgrades.")
from importlib.metadata import version
print("transformers", version("transformers"))

all packages already present – no installs, no upgrades.
transformers 5.0.0


In [2]:
REPO_ID = "Shanmuk4622/ai-detection-dataset-v2"
import os
from huggingface_hub import hf_hub_download
def load_token():
    try:
        from kaggle_secrets import UserSecretsClient
        t = UserSecretsClient().get_secret("HF_TOKEN")
        if t: return t.strip()
    except Exception: pass
    for k in ("HF_TOKEN", "HUGGINGFACE_TOKEN", "HUGGING_FACE_HUB_TOKEN"):
        if os.environ.get(k): return os.environ[k].strip()
    import getpass; return getpass.getpass("HF write token: ").strip()
TOKEN = load_token()
lib = hf_hub_download(REPO_ID, "ai_detect_common.py", repo_type="dataset", token=TOKEN)
import importlib.util
spec = importlib.util.spec_from_file_location("ai_detect_common", lib)
C = importlib.util.module_from_spec(spec); spec.loader.exec_module(C)
cfg = C.read_config(REPO_ID, TOKEN)
MASTER_SEED = cfg.get("master_seed", 42)   # change master_seed in NB00 to reseed everything
print(f"library {C.PIPELINE_VERSION} loaded | MASTER_SEED={MASTER_SEED}")

ai_detect_common.py: 0.00B [00:00, ?B/s]

config.json: 0.00B [00:00, ?B/s]

library 1.2 loaded | MASTER_SEED=42


## Load BLIP-2  (native transformers – no patches needed)

In [3]:
import torch, re
from transformers import Blip2Processor, Blip2ForConditionalGeneration, CLIPTokenizer

device, dtype = C.get_device_dtype()
assert device == "cuda", "Turn the GPU ON for NB02."

CAPTION_MODEL = cfg["caption_model"]   # Salesforce/blip2-opt-2.7b
proc = Blip2Processor.from_pretrained(CAPTION_MODEL)
try:
    blip = Blip2ForConditionalGeneration.from_pretrained(
        CAPTION_MODEL, dtype=dtype).to(device).eval()
except TypeError:
    blip = Blip2ForConditionalGeneration.from_pretrained(
        CAPTION_MODEL, torch_dtype=dtype).to(device).eval()
clip_tok = CLIPTokenizer.from_pretrained("openai/clip-vit-large-patch14")
MAXTOK = cfg.get("caption_max_tokens", 75)
print(f"BLIP-2 loaded on {device} | MASTER_SEED={MASTER_SEED}")

@torch.no_grad()
def caption_image(pil):
    inputs = proc(images=pil, return_tensors="pt").to(device, dtype)
    out = blip.generate(**inputs, max_new_tokens=60, num_beams=5,
                        do_sample=False, repetition_penalty=1.3)
    return proc.batch_decode(out, skip_special_tokens=True)[0].strip()

_LEADINS = ("the image shows ","the image depicts ","this image shows ","this image depicts ",
            "the image is ","the photo shows ","an image of ","a photo of ",
            "the image captures ","this is ","a picture of ","there is ","there are ")
def enhance_caption(raw, max_tokens=MAXTOK):
    s = re.sub(r"\s+"," ",(raw or "").strip()); low = s.lower()
    for p in _LEADINS:
        if low.startswith(p): s = s[len(p):]; s=(s[:1].upper()+s[1:]) if s else s; break
    if not s: s = "a photograph"
    ids = clip_tok(s, add_special_tokens=False)["input_ids"]
    if len(ids) > max_tokens:
        s = clip_tok.decode(ids[:max_tokens]).strip()
        ids = clip_tok(s, add_special_tokens=False)["input_ids"]
    return s, len(ids)

# smoke test
_shards = C.list_shards(REPO_ID, "real/", TOKEN)
if _shards:
    _loc = C.hf_hub_download(REPO_ID, _shards[0], repo_type="dataset", token=TOKEN)
    _t   = C.pq.read_table(_loc, columns=["image"])
    _raw = caption_image(C.decode_png(_t.column("image")[0].as_py()))
    print("sample raw :", repr(_raw))
    print("sample enhanced:", enhance_caption(_raw))

processor_config.json:   0%|          | 0.00/68.0 [00:00<?, ?B/s]

preprocessor_config.json:   0%|          | 0.00/432 [00:00<?, ?B/s]

The image processor of type `BlipImageProcessor` is now loaded as a fast processor by default, even if the model checkpoint was saved with a slow processor. This is a breaking change and may produce slightly different outputs. To continue using the slow processor, instantiate this class with `use_fast=False`. 


config.json: 0.00B [00:00, ?B/s]

tokenizer_config.json:   0%|          | 0.00/882 [00:00<?, ?B/s]

...

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/389 [00:00<?, ?B/s]

BLIP-2 loaded on cuda | MASTER_SEED=42


real/real-70beac4d-00000.parquet:   0%|          | 0.00/222M [00:00<?, ?B/s]

sample raw : 'a lunch box with food in it'
sample enhanced: ('a lunch box with food in it', 7)


## Caption every real image (resume-safe, 20-min commits)

In [4]:
api  = C.HfApi(token=TOKEN)
done = C.reconcile_ids(REPO_ID, "captions/", TOKEN)
print("already captioned:", len(done))

cwriter = C.ShardWriter(api, REPO_ID, "captions/", schema=C.CAPTION_SCHEMA,
                        commit_interval=cfg["commit_interval_seconds"],
                        max_rows=cfg["batch_size"], token=TOKEN)
processed = 0
try:
    for f in C.list_shards(REPO_ID, "real/", TOKEN):
        local = C.hf_hub_download(REPO_ID, f, repo_type="dataset", token=TOKEN)
        t = C.pq.read_table(local, columns=["image_id","image"])
        iids, imgs = t.column("image_id").to_pylist(), t.column("image")
        for k, iid in enumerate(iids):
            if iid in done: continue
            pil = C.decode_png(imgs[k].as_py())
            raw = caption_image(pil)
            cap, ntok = enhance_caption(raw)
            cwriter.add(dict(source_real_id=iid, caption=cap, raw_caption=raw,
                             caption_model=CAPTION_MODEL, caption_task=cfg["caption_task"],
                             n_tokens=ntok, created_utc=C.now_utc()))
            done.add(iid); processed += 1
            if processed % 200 == 0: print(f"  captioned {processed} (total ~{len(done)})")
            cwriter.maybe_flush()
finally:
    cwriter.close()
print("captioning complete; this run:", processed)

captions/captions-906881cf-00000.parquet:   0%|          | 0.00/23.2k [00:00<?, ?B/s]

captions/captions-906881cf-00001.parquet:   0%|          | 0.00/23.1k [00:00<?, ?B/s]

captions/captions-906881cf-00002.parquet:   0%|          | 0.00/23.3k [00:00<?, ?B/s]

captions/captions-906881cf-00003.parquet:   0%|          | 0.00/23.5k [00:00<?, ?B/s]

captions/captions-906881cf-00004.parquet:   0%|          | 0.00/23.1k [00:00<?, ?B/s]

...

New Data Upload: |          |  0.00B /  0.00B            

  committed 500 rows (3000 total) -> captions/captions-906ec70b-00005.parquet
  captioned 3000 (total ~9858)


Processing Files (0 / 0): |          |  0.00B /  0.00B            

New Data Upload: |          |  0.00B /  0.00B            

  committed 142 rows (3142 total) -> captions/captions-906ec70b-00006.parquet
captioning complete; this run: 3142


## Verifier

In [5]:
from collections import Counter
real_ids = set()
for f in C.list_shards(REPO_ID, "real/", TOKEN):
    local = C.hf_hub_download(REPO_ID, f, repo_type="dataset", token=TOKEN)
    real_ids.update(C.pq.read_table(local, columns=["image_id"]).column("image_id").to_pylist())
cap_ids, empties, toolong = [], 0, 0
for f in C.list_shards(REPO_ID, "captions/", TOKEN):
    local = C.hf_hub_download(REPO_ID, f, repo_type="dataset", token=TOKEN)
    t = C.pq.read_table(local, columns=["source_real_id","caption","n_tokens"])
    cap_ids += t.column("source_real_id").to_pylist()
    empties += sum(1 for c in t.column("caption").to_pylist() if not c or not str(c).strip())
    toolong += sum(1 for n in t.column("n_tokens").to_pylist() if n and n > 77)
dups    = [k for k,v in Counter(cap_ids).items() if v>1]
missing = real_ids - set(cap_ids)
print(f"real={len(real_ids)}  captioned={len(set(cap_ids))}  missing={len(missing)}"
      f"  dups={len(dups)}  empty={empties}  over_77_tokens={toolong}")
print("RESULT:", "PASS" if not (missing or dups or empties or toolong) else "CHECK ABOVE")

captions/captions-906ec70b-00000.parquet:   0%|          | 0.00/24.0k [00:00<?, ?B/s]

captions/captions-906ec70b-00001.parquet:   0%|          | 0.00/24.4k [00:00<?, ?B/s]

captions/captions-906ec70b-00002.parquet:   0%|          | 0.00/23.5k [00:00<?, ?B/s]

captions/captions-906ec70b-00003.parquet:   0%|          | 0.00/24.3k [00:00<?, ?B/s]

captions/captions-906ec70b-00004.parquet:   0%|          | 0.00/24.4k [00:00<?, ?B/s]

captions/captions-906ec70b-00005.parquet:   0%|          | 0.00/24.3k [00:00<?, ?B/s]

captions/captions-906ec70b-00006.parquet:   0%|          | 0.00/9.19k [00:00<?, ?B/s]

real=10000  captioned=10000  missing=0  dups=0  empty=0  over_77_tokens=0
RESULT: PASS
